In [ ]:
import numpy as np
import pandas as pd


In [ ]:
states = ('phishing', 'legit')
signals = ('flagged', 'not_flagged')
prior_probs = np.array([0.10, 0.90])
phishing_probs = np.array([0.85, 0.15])
legit_probs = np.array([0.05, 0.95])


In [ ]:
def draw_prior(states, probs, N, rng):
    return rng.choice(states, size=N, p=probs)

def draw_model(signals, state_vec, phishing_probs, legit_probs, rng):
    obs = np.empty(state_vec.shape[0], dtype=object)
    mask_phish = (state_vec == 'phishing')
    obs[mask_phish] = rng.choice(signals, size=mask_phish.sum(), p=phishing_probs)
    obs[~mask_phish] = rng.choice(signals, size=(~mask_phish).sum(), p=legit_probs)
    return obs

def draw_joint(signals, states, prior_probs, phishing_probs, legit_probs, N, rng):
    s = draw_prior(states, prior_probs, N, rng)
    f = draw_model(signals, s, phishing_probs, legit_probs, rng)
    return s, f

def simulator(signals, states, prior_probs, phishing_probs, legit_probs, num_sims=10000, seed=0):
    rng = np.random.default_rng(seed)
    return draw_joint(signals, states, prior_probs, phishing_probs, legit_probs, num_sims, rng)


In [ ]:
num_sims = 100_000
S, F = simulator(signals, states, prior_probs, phishing_probs, legit_probs, num_sims=num_sims, seed=42)


In [ ]:
joint_labels = np.char.add(np.char.add(S.astype(str), '_'), F.astype(str))
scenarios, counts = np.unique(joint_labels, return_counts=True)
for s, c in zip(scenarios, counts):
    print(f'{s} occurred with a probability of {c / num_sims:.4f}.')


legit_flagged occurred with a probability of 0.0445.
legit_not_flagged occurred with a probability of 0.8563.
phishing_flagged occurred with a probability of 0.0841.
phishing_not_flagged occurred with a probability of 0.0151.


In [ ]:
df = pd.DataFrame({'state': S, 'signal': F})
counts_tab = pd.crosstab(df['state'], df['signal']).reindex(index=states, columns=signals, fill_value=0)
approx_joint = counts_tab / num_sims
approx_joint


signal,flagged,not_flagged
state,,
phishing,0.08407,0.01509
legit,0.04455,0.85629
